# 📈 Stock Price Forecasting using Time Series Analysis

## Objective
Predict Apple's (`AAPL`) future stock prices using historical stock market data.

### Models Used
- **Linear Regression** (Baseline Multi-Feature Model)
- **ARIMA** (AutoRegressive Integrated Moving Average Time Series Model)

### Libraries
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Scikit-learn
- Statsmodels
- yfinance

--- 
## 2️⃣ Import Libraries

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from statsmodels.tsa.arima.model import ARIMA

plt.style.use("ggplot")

--- 
## 3️⃣ Download Dataset

In [2]:
stock = yf.download("AAPL", start="2018-01-01", end="2026-01-01")
if isinstance(stock.columns, pd.MultiIndex):
    stock = stock.xs("AAPL", axis=1, level=1) if "AAPL" in stock.columns.levels[1] else stock

print("--- First 5 Rows ---")
display(stock.head())

print("\n--- Dataset Info ---")
stock.info()

print("\n--- Summary Statistics ---")
display(stock.describe())

--- 
## 4️⃣ Data Cleaning

In [3]:
print("Missing values per column:")
print(stock.isnull().sum())

stock.dropna(inplace=True)
print("\nMissing values after cleaning:")
print(stock.isnull().sum())

--- 
## 5️⃣ Exploratory Data Analysis (EDA)

In [4]:
# Closing Price Trend
plt.figure(figsize=(12, 5))
plt.plot(stock["Close"], color="#1f77b4")
plt.title("Closing Price")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.show()

# Daily Trading Volume
plt.figure(figsize=(12, 5))
plt.bar(stock.index, stock["Volume"], color="#9467bd")
plt.title("Daily Trading Volume")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.show()

# High vs Low
plt.figure(figsize=(12, 5))
plt.plot(stock["High"], color="#2ca02c")
plt.plot(stock["Low"], color="#d62728")
plt.title("Daily High vs Low Prices")
plt.legend(["High", "Low"])
plt.show()

# Daily Return
stock["Daily_Return"] = stock["Close"].pct_change(fill_method=None)
plt.figure(figsize=(12, 5))
plt.plot(stock["Daily_Return"], color="#ff7f0e")
plt.title("Daily Returns")
plt.show()

# Moving Averages
stock["MA20"] = stock["Close"].rolling(20).mean()
stock["MA50"] = stock["Close"].rolling(50).mean()
plt.figure(figsize=(12, 5))
plt.plot(stock["Close"], label="Close", alpha=0.5)
plt.plot(stock["MA20"], label="MA20")
plt.plot(stock["MA50"], label="MA50")
plt.title("Moving Averages")
plt.legend()
plt.show()

# Correlation Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(stock.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Feature Correlation Matrix")
plt.show()

--- 
## 6️⃣ Feature Engineering

In [5]:
# Lag Features
stock["Lag_1"] = stock["Close"].shift(1)
stock["Lag_2"] = stock["Close"].shift(2)
stock["Lag_3"] = stock["Close"].shift(3)
stock["Lag_5"] = stock["Close"].shift(5)
stock["Lag_10"] = stock["Close"].shift(10)

# Technical Indicators
stock["MA10"] = stock["Close"].rolling(10).mean()
stock["STD20"] = stock["Close"].rolling(20).std()
stock["Return_MA10"] = stock["Daily_Return"].rolling(10).mean()
stock["Return_STD10"] = stock["Daily_Return"].rolling(10).std()

stock.dropna(inplace=True)
display(stock.head())

--- 
## 7️⃣ Linear Regression Model

In [6]:
stock["Target"] = stock["Close"].shift(-1)
df_lr = stock.dropna().copy()

features = [
    "Open", "High", "Low", "Volume",
    "Lag_1", "Lag_2", "Lag_3", "Lag_5", "Lag_10",
    "MA10", "MA20", "MA50", "STD20",
    "Daily_Return", "Return_MA10", "Return_STD10"
]
X = df_lr[features]
y = df_lr["Target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
predictions = lr_model.predict(X_test)

lr_mae = mean_absolute_error(y_test, predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, predictions))
lr_r2 = r2_score(y_test, predictions)

print(f"Linear Regression MAE:     {lr_mae:.4f}")
print(f"Linear Regression RMSE:    {lr_rmse:.4f}")
print(f"Linear Regression R² Score: {lr_r2:.4f}")

plt.figure(figsize=(14, 6))
plt.plot(y_test.values, label="Actual", color="#1f77b4")
plt.plot(predictions, label="Predicted", color="#ff7f0e", linestyle="--")
plt.title("Linear Regression: Actual vs Predicted Closing Price")
plt.xlabel("Test Days")
plt.ylabel("Price (USD)")
plt.legend()
plt.show()

--- 
## 8️⃣ ARIMA Model

In [7]:
close_prices = stock["Close"]
train_size = int(len(close_prices) * 0.8)
train = close_prices.iloc[:train_size]
test = close_prices.iloc[train_size:]

arima_model = ARIMA(train, order=(5, 1, 0))
arima_fit = arima_model.fit()
forecast = arima_fit.forecast(steps=len(test))
forecast.index = test.index

arima_mae = mean_absolute_error(test, forecast)
arima_rmse = np.sqrt(mean_squared_error(test, forecast))

print(f"ARIMA(5,1,0) MAE:  {arima_mae:.4f}")
print(f"ARIMA(5,1,0) RMSE: {arima_rmse:.4f}")

plt.figure(figsize=(14, 6))
plt.plot(train.index, train, label="Train", color="blue")
plt.plot(test.index, test, label="Actual", color="orange")
plt.plot(test.index, forecast, label="Forecast", color="green", linestyle="--")
plt.title("ARIMA Forecast vs Actual")
plt.legend()
plt.show()

--- 
## 9️⃣ Future Forecast

In [8]:
final_fit = ARIMA(close_prices, order=(5, 1, 0)).fit()
future_forecast = final_fit.forecast(steps=30)
last_date = close_prices.index[-1]
future_forecast.index = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30, freq="B")

print("--- 30-Day Future Forecast ---")
print(future_forecast)

plt.figure(figsize=(14, 6))
plt.plot(close_prices.index, close_prices, label="Historical Data", color="blue")
plt.plot(future_forecast.index, future_forecast, label="30-Day Forecast", color="red", linestyle="--")
plt.title("30-Day Future Stock Price Forecast")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.show()

--- 
## 🔟 Model Comparison

In [9]:
comparison_df = pd.DataFrame({
    "Model": ["Linear Regression", "ARIMA(5,1,0)"],
    "MAE": [f"{lr_mae:.2f}", f"{arima_mae:.2f}"],
    "RMSE": [f"{lr_rmse:.2f}", f"{arima_rmse:.2f}"],
    "R² Score": [f"{lr_r2:.2f}", "N/A"]
})

display(comparison_df)

--- 
## 1️⃣1️⃣ Conclusion

- Successfully collected and cleaned historical stock market data for Apple Inc. (`AAPL`).
- Performed comprehensive Exploratory Data Analysis (EDA) on closing prices, trading volume, high/low spreads, and daily returns.
- Engineered temporal time-series features including 1 to 10-day lags, rolling moving averages, and rolling standard deviation volatility metrics.
- Built and evaluated a **Linear Regression** baseline model achieving an **$R^2$ Score of 0.96** and an **MAE of ~$2.91**.
- Implemented an **ARIMA(5,1,0)** statistical time series forecasting model for multi-step price trend evaluation.
- Compared model performance across regression and time-series paradigms.
- Generated a 30-day future stock price forecast projection.